# Module 6.6 — CI/CD & DevOps

### Python Mastery · Track 6: Testing, Tooling & DevOps

Continuous integration and continuous delivery (CI/CD) automate the checks and steps that turn code into a running, trustworthy system. This module covers CI workflow files, testing across multiple environments with tox and nox, containerising an application with Docker, and handling configuration and secrets.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- CI runs on a hosting platform and Docker requires a daemon, so those pieces are demonstrated by writing the real configuration files and validating their structure here, rather than executing the platform itself. The configuration-handling examples (`.env` parsing) run fully.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Explain what continuous integration does and write a CI workflow file.
2. Describe testing across versions with tox and nox.
3. Write a `Dockerfile` to containerise a Python application.
4. Manage configuration and secrets with environment variables and `.env` files.
5. Describe how these pieces fit into a delivery pipeline.

**Prerequisites:** Tracks 1 to 5, and Modules 6.1 to 6.5.

---

## Part 1 · Continuous Integration

**Continuous integration** means that every change pushed to a shared repository is automatically built and checked: tests are run, types are checked, and the linter and formatter are verified. If anything fails, the change is flagged before it can be merged. This catches problems early and keeps the main branch always working.

CI runs on a hosting platform (GitHub Actions, GitLab CI, and others), configured by a file in the repository. The cell below writes a typical GitHub Actions workflow that lints, type-checks, and tests on every push.

In [ ]:
import os
os.makedirs(".github/workflows", exist_ok=True)

workflow = """name: CI

on: [push, pull_request]          # run on every push and pull request

jobs:
  check:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4              # get the code
      - uses: actions/setup-python@v5          # install Python
        with:
          python-version: "3.12"
      - run: pip install ruff mypy pytest      # install tools
      - run: ruff check .                      # lint
      - run: mypy .                            # type-check
      - run: pytest -q                         # test
"""
with open(".github/workflows/ci.yml", "w") as f:
    f.write(workflow)

print(open(".github/workflows/ci.yml").read())

We can confirm the workflow is well-formed YAML by parsing it, which is also what the platform does before running it. A syntactically valid file is the first requirement for a working pipeline.

In [ ]:
import yaml

with open(".github/workflows/ci.yml") as f:
    workflow = yaml.safe_load(f)

print("workflow name:", workflow["name"])
# Note: PyYAML follows YAML 1.1, which reads the unquoted key 'on' as the boolean
# True (a well-known gotcha), so the triggers are stored under the key True.
print("triggers:", workflow.get("on", workflow.get(True)))
print("jobs:", list(workflow["jobs"].keys()))
steps = workflow["jobs"]["check"]["steps"]
print("number of steps:", len(steps))
print("commands run:", [s["run"] for s in steps if "run" in s])

## Part 2 · Multi-Environment Testing with tox and nox

A library should work across several Python versions and dependency sets. **tox** and **nox** automate running your test suite in multiple isolated environments, so you catch version-specific problems before users do. tox is configured declaratively; nox is configured in Python, which some find more flexible.

The cell below writes a typical `tox.ini` that runs tests on three Python versions.

In [ ]:
tox_ini = """[tox]
envlist = py39, py310, py311, py312     # test on several Python versions

[testenv]
deps = pytest                            # install test dependencies in each env
commands = pytest -q                     # run the suite in each env
"""
with open("tox.ini", "w") as f:
    f.write(tox_ini)

print(open("tox.ini").read())
print("Run all environments with: tox")
print("nox is the Python-configured equivalent, defined in a noxfile.py.")

## Part 3 · Containerising with Docker

A **container** packages an application together with its exact environment, so it runs the same on a developer's laptop, a test server, and in production. A `Dockerfile` is the recipe: it starts from a base image, copies in the code, installs dependencies, and defines the start command. Building it requires the Docker daemon, so here we write and examine a representative `Dockerfile`.

In [ ]:
dockerfile = """# Start from a small official Python image.
FROM python:3.12-slim

# Set the working directory inside the container.
WORKDIR /app

# Copy and install dependencies first, so this layer is cached when only code changes.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the application code.
COPY . .

# The command that runs when the container starts.
CMD ["python", "main.py"]
"""
with open("Dockerfile", "w") as f:
    f.write(dockerfile)

print(open("Dockerfile").read())
print("Build with:  docker build -t myapp .")
print("Run with:    docker run myapp")

The ordering matters: copying and installing dependencies **before** copying the rest of the code lets Docker reuse the cached dependency layer when only the application code changes, making rebuilds much faster. This layer-caching principle is central to writing efficient Dockerfiles.

## Part 4 · Configuration and Secrets

Applications need configuration (database addresses, feature flags) and **secrets** (passwords, API keys). The firm rule is to keep these **out of the source code**: secrets in code can leak through version control and cannot vary by environment. The standard approach is **environment variables**, often loaded from a `.env` file that is excluded from version control.

The cell below writes a `.env` file and reads it with a small parser, then through `os.environ`, showing how an application would pick up its configuration.

In [ ]:
import os

# A .env file holds key=value settings; it is NEVER committed to version control.
with open(".env", "w") as f:
    f.write("DATABASE_URL=postgres://localhost/mydb\n")
    f.write("DEBUG=true\n")
    f.write("API_KEY=secret-do-not-share\n")

# A minimal parser, illustrating what libraries such as python-dotenv do.
def load_env(path):
    settings = {}
    for line in open(path):
        line = line.strip()
        if line and not line.startswith("#"):
            key, value = line.split("=", 1)
            settings[key] = value
    return settings

config = load_env(".env")
print("loaded keys:", list(config))
print("DATABASE_URL:", config["DATABASE_URL"])
print("DEBUG flag:", config["DEBUG"] == "true")

In [ ]:
import os

# In practice, settings are placed into the environment and read via os.environ,
# so the same code works whether values come from a .env file or the real environment.
os.environ["APP_MODE"] = "production"            # normally set outside the program

mode = os.environ.get("APP_MODE", "development") # read with a sensible default
missing = os.environ.get("UNSET_VALUE", "fallback")
print("mode:", mode)
print("missing value uses default:", missing)
print("Add .env to .gitignore so secrets are never committed.")

## Part 5 · The Delivery Pipeline

Putting it together, a typical pipeline runs on each change: CI checks out the code, installs dependencies, and runs the linter, type checker, and tests across the supported environments. If all pass, **continuous delivery** can take over, building the package or container image and deploying it. Each stage is a gate; a failure stops the pipeline and alerts the team, so only verified code reaches production. The cell summarises the stages.

In [ ]:
pipeline = [
    ("Commit / push", "developer sends a change to the repository"),
    ("CI: install", "set up Python and install dependencies"),
    ("CI: lint", "ruff checks style and likely bugs"),
    ("CI: type-check", "mypy verifies type annotations"),
    ("CI: test", "pytest runs the suite (across versions via tox/nox)"),
    ("CD: build", "build a wheel or a Docker image"),
    ("CD: deploy", "release to staging, then production"),
]
print("A typical CI/CD pipeline:")
for stage, description in pipeline:
    print(f"  {stage:16} -> {description}")
print()
print("Each stage is a gate: a failure stops the pipeline before bad code ships.")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A minimal CI workflow (Easy)

In [ ]:
import os
os.makedirs(".github/workflows", exist_ok=True)
workflow = """name: Test
on: [push]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install pytest
      - run: pytest -q
"""
with open(".github/workflows/test.yml", "w") as f:
    f.write(workflow)
print(open(".github/workflows/test.yml").read())

### Example 2 — Reading an environment variable (Easy)

In [ ]:
import os
os.environ["GREETING"] = "hello"
print("value:", os.environ.get("GREETING"))
print("with default:", os.environ.get("MISSING", "default value"))

### Example 3 — Validating a workflow file (Medium)

In [ ]:
import yaml

workflow_text = """
name: Build
on: [push, pull_request]
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - run: echo building
      - run: echo testing
"""
parsed = yaml.safe_load(workflow_text)
print("valid YAML:", isinstance(parsed, dict))
print("name:", parsed["name"])
print("steps:", [s["run"] for s in parsed["jobs"]["build"]["steps"]])

### Example 4 — Parsing a .env file (Medium)

In [ ]:
with open("ex4.env", "w") as f:
    f.write("# configuration\n")
    f.write("HOST=localhost\n")
    f.write("PORT=5432\n")
    f.write("\n")                      # blank line, should be ignored
    f.write("DEBUG=false\n")

settings = {}
for line in open("ex4.env"):
    line = line.strip()
    if line and not line.startswith("#"):
        key, value = line.split("=", 1)
        settings[key] = value

print(settings)
print("port as int:", int(settings["PORT"]))

### Example 5 — A tox configuration for several versions (Difficult)

In [ ]:
import configparser

tox_text = """[tox]
envlist = py310, py311, py312

[testenv]
deps =
    pytest
    pytest-cov
commands =
    pytest --cov=mypackage -q
"""
with open("ex5_tox.ini", "w") as f:
    f.write(tox_text)

# tox.ini is INI format; we can parse it to confirm the environments.
parser = configparser.ConfigParser()
parser.read("ex5_tox.ini")
print("environments:", parser["tox"]["envlist"])
print("test command(s):", parser["testenv"]["commands"].strip())

### Example 6 — A multi-stage Dockerfile and its structure (Difficult)

In [ ]:
dockerfile = """# Stage 1: build (install dependencies into a target directory).
FROM python:3.12-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/deps -r requirements.txt

# Stage 2: runtime (copy only what is needed, for a smaller final image).
FROM python:3.12-slim
WORKDIR /app
COPY --from=builder /deps /deps
COPY . .
ENV PYTHONPATH=/deps
CMD ["python", "main.py"]
"""
with open("Dockerfile.multistage", "w") as f:
    f.write(dockerfile)

# Count the build stages (each FROM begins one).
stages = [line for line in dockerfile.splitlines() if line.startswith("FROM")]
print("number of build stages:", len(stages))
print("multi-stage builds keep the final image small by discarding build-only files")

---

## Exercises

**Exercise 1 (Easy).** Write a GitHub Actions workflow file that, on push, sets up Python 3.12 and runs `pytest -q`. Print the file.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Set an environment variable `ENVIRONMENT` to `"staging"` and read it back with a default of `"local"`. Print the value.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a `.env` file with three settings (including a comment line and a blank line), parse it into a dictionary ignoring comments and blanks, and print the dictionary.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Write a `tox.ini` testing on `py311` and `py312` with `pytest` as the dependency, then parse it with `configparser` and print the environment list.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a `Dockerfile` for a Python 3.12 app that installs from `requirements.txt`, copies the code, and runs `app.py`. Then count and print how many instruction lines (those starting with an uppercase keyword such as FROM, RUN, COPY, CMD) it contains.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import os
os.makedirs(".github/workflows", exist_ok=True)
workflow = """name: Test
on: [push]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install pytest
      - run: pytest -q
"""
with open("sol1_ci.yml", "w") as f:
    f.write(workflow)
print(open("sol1_ci.yml").read())

**Solution 2**

In [ ]:
import os
os.environ["ENVIRONMENT"] = "staging"
print(os.environ.get("ENVIRONMENT", "local"))

**Solution 3**

In [ ]:
with open("sol3.env", "w") as f:
    f.write("# settings\nNAME=app\n\nVERSION=1.0\nDEBUG=true\n")

config = {}
for line in open("sol3.env"):
    line = line.strip()
    if line and not line.startswith("#"):
        k, v = line.split("=", 1)
        config[k] = v
print(config)

**Solution 4**

In [ ]:
import configparser
tox_ini = """[tox]
envlist = py311, py312

[testenv]
deps = pytest
commands = pytest -q
"""
with open("sol4_tox.ini", "w") as f:
    f.write(tox_ini)
parser = configparser.ConfigParser()
parser.read("sol4_tox.ini")
print(parser["tox"]["envlist"])

**Solution 5**

In [ ]:
dockerfile = """FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["python", "app.py"]
"""
with open("sol5_Dockerfile", "w") as f:
    f.write(dockerfile)

keywords = ("FROM", "WORKDIR", "COPY", "RUN", "CMD", "ENV", "EXPOSE")
instructions = [l for l in dockerfile.splitlines() if l.split(" ")[0] in keywords]
print("instruction lines:", len(instructions))

---

## Summary and Key Points

- Continuous integration automatically builds and checks every change (lint, type-check, test) before it merges, keeping the main branch working; it is configured by a workflow file on a hosting platform.
- tox and nox run the test suite across several Python versions in isolated environments, catching version-specific problems.
- A `Dockerfile` packages an application with its exact environment so it runs identically everywhere; install dependencies before copying code to benefit from layer caching.
- Keep configuration and secrets out of source code: use environment variables, often loaded from a `.env` file that is excluded from version control, and read with `os.environ.get` plus defaults.
- A delivery pipeline chains these stages as gates, install, lint, type-check, test, build, deploy, so only verified code reaches production.

### Next module: 6.7 — Security Best Practices

The next module covers writing safer code: validating input, avoiding dangerous functions, generating secure random values, safe subprocess use, hashing, and scanning dependencies.